# 让 AI 访谈更可控的方法

目标：让 AI 访谈更像真人对话，同时保持可控。

核心观点：**通常不需要先做模型微调**。先从提示工程、工作流、评估迭代入手，成本低、见效快。微调放在后期，用来固化风格或领域习惯。

## 0. 当前系统是怎么工作的

1. 用户配置模型提供商、API Key、模型。
2. 输入研究目标。
3. 系统把研究目标包装成系统提示词，让 AI 生成第一个问题。
4. 每次用户回答后，把完整对话历史再发给 AI，让它生成下一个问题。
5. 用户点击结束，系统把完整对话发给 AI，让它生成结构化报告。

可控性不足的地方：AI 自由发挥较多，没有明确的阶段控制，追问策略单一。

## 1. 提示工程（Prompt Engineering）

最轻量、最先尝试的方法。

### 1.1 系统提示词更具体
- 定义角色：经验丰富的定性研究主持人。
- 定义阶段：开场 → 背景 → 核心探索 → 深度追问 → 收尾。
- 定义语气：对话式、尊重、不评价。
- 定义约束：一次只问一个问题，不总结、不分析。

### 1.2 Few-shot 示例
在提示词里加入 3-5 段高质量真人访谈片段，AI 会模仿这种节奏和追问方式。

### 1.3 结构化输出
要求 AI 每轮返回 JSON，包含：
- `question`：下一个问题
- `stage`：当前阶段
- `reason`：为什么问这个问题

这样后端可以校验、干预，甚至拒绝不符合阶段的问题。

In [ ]:
# 示例：更具体的系统提示词
system_prompt = """
你是一位经验丰富的定性研究访谈主持人。
研究目标：{goal}

访谈阶段：
1. 开场：建立信任，说明目的，问一个轻松的开放问题。
2. 背景：了解受访者的基本情况和使用场景。
3. 核心探索：围绕研究目标深入挖掘。
4. 深度追问：对关键回答追问动机、感受、具体例子。
5. 收尾：总结确认，感谢受访者。

规则：
- 一次只问一个问题。
- 语气自然、对话式，避免像问卷。
- 追问时先说一小句对回答的理解，再问下一个问题。
- 如果受访者回答简短， gently 请他举个例子。
- 不输出分析、总结、bullet list。
- 输出必须是 JSON：{{"question": "...", "stage": "...", "reason": "..."}}
"""
print(system_prompt[:200])

## 2. 状态机 / 工作流（State Machine）

把访谈拆成明确的阶段，每轮根据上下文判断当前阶段，再调用对应提示词。

优点：
- 可控性强，每个阶段有固定策略。
- 可以插入口袋问题（fallback questions）。
- 容易评估和调试。

实现方式：
- 硬编码规则：轮数、关键词、情绪检测。
- 让 AI 自己判断阶段：每轮先调一个 cheap 模型做阶段分类，再调主模型生成问题。
- 混合：关键节点用规则，其余交给 AI。

In [ ]:
# 示例：简单的阶段判断函数
def infer_stage(messages, goal):
    n = len(messages)
    if n <= 1:
        return 'opening'
    if n <= 3:
        return 'background'
    if n <= 6:
        return 'core_exploration'
    if any('为什么' in m['text'] or '原因' in m['text'] for m in messages[-2:] if m['role'] == 'user'):
        return 'deep_probing'
    return 'closing'

messages = [
    {'role': 'assistant', 'text': '你好，能简单介绍一下自己吗？'},
    {'role': 'user', 'text': '我是一名产品经理，平时做用户调研。'},
]
print(infer_stage(messages, '了解用户研究方法'))

## 3. 检索增强（RAG）

把访谈提纲、过往优秀访谈、领域知识做成向量库。每次生成问题前，先检索相关参考，再让 AI 参照生成。

适用场景：
- 公司有一套访谈方法论（如 Jobs-to-be-Done、JTBD）。
- 想复用某次特别成功的访谈风格。
- 让 AI 自动引用特定概念或框架。

不需要自己训练模型，只需要好的数据和向量检索。

## 4. 函数调用 / 工具（Function Calling / Tools）

让 AI 在特定节点调用工具，而不是纯文本自由发挥。

可以定义的工具：
- `ask_question(question, stage)`：正常提问。
- `probe(reason, target)`：追问某个点。
- `summarize_and_confirm()`：阶段小结并请用户确认。
- `end_interview()`：结束访谈。

后端收到函数调用后，可以执行、校验、或改写。这样 AI 的输出就被限制在一组预定义动作里。

In [ ]:
# 示例：工具定义（伪代码，OpenAI / Anthropic 格式类似）
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'ask_question',
            'description': '向受访者提出下一个问题',
            'parameters': {
                'type': 'object',
                'properties': {
                    'question': {'type': 'string', 'description': '问题内容'},
                    'stage': {'type': 'string', 'enum': ['opening', 'background', 'core_exploration', 'deep_probing', 'closing']},
                    'reason': {'type': 'string', 'description': '为什么问这个问题'}
                },
                'required': ['question', 'stage', 'reason']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'end_interview',
            'description': '结束访谈并感谢受访者',
            'parameters': {'type': 'object', 'properties': {}}
        }
    }
]
print(len(tools))

## 5. 评估与迭代（Eval Loop）

没有评估，就不知道哪种方法更好。

评估维度：
- 问题是否自然、不机械？
- 追问是否切中要点？
- 是否跑题或过早下结论？
- 受访者回答长度是否合适？
- 最终报告是否覆盖研究目标？

方法：
- 人工抽检 10-20 段对话。
- 用另一个 AI 模型当评委，按维度打分。
- 记录 bad case，改提示词或工作流。
- 做 A/B 测试：对比不同提示词版本。

In [ ]:
# 示例：简单的评估打分表
rubric = {
    'naturalness': '问题是否像真人对话，不生硬',
    'relevance': '问题是否紧扣研究目标',
    'probing': '是否基于用户回答做了有效追问',
    'single_question': '一次是否只问一个问题',
    'no_bias': '是否没有引导性或偏见性语言',
}
for k, v in rubric.items():
    print(f'{k}: {v}')

## 6. 模型微调（Fine-tuning）

什么时候才考虑微调？
- 提示词和工作流已经做到 80 分，但某些风格、领域术语、追问节奏始终调不好。
- 积累了大量高质量真人访谈数据，想复制某位优秀主持人的风格。
- 需要模型稳定地遵守特定方法论（如 JTBD、五问法）。

微调方法：

| 方法 | 说明 | 成本 | 适用场景 |
|---|---|---|---|
| **SFT（监督微调）** | 用高质量问答对训练模型 | 中等 | 模仿特定访谈风格、固定格式 |
| **LoRA / QLoRA** | 只训练少量参数，效率更高 | 低 | 快速实验、资源有限 |
| **RLHF / DPO** | 用人类偏好数据优化 | 高 | 让模型学会更自然的对话偏好 |
| **全量微调** | 训练整个模型 | 很高 | 追求最佳效果，通常没必要 |

注意：微调需要成对的训练数据，比如（对话上下文 → 优秀主持人下一句话）。数据质量比数据量更重要。

## 7. 推荐落地路线

1. **先优化系统提示词**：加入阶段定义、few-shot 示例、结构化输出。
2. **引入状态机**：用规则 + 小模型做阶段判断，控制访谈节奏。
3. **加入 RAG**：检索访谈提纲和优秀案例。
4. **用函数调用限制输出**：把提问、追问、结束变成可调用的工具。
5. **建立评估循环**：收集 bad case，打分，迭代。
6. **必要时微调**：当提示词 + 工作流的天花板明显，且有足够数据时，再考虑 LoRA 或 SFT。

对于 ow-text 当前这个项目，建议先做第 1-2 步，就能明显提升可控性。